# 🏷️ P2 · Labeln und prüfen

**Projekt-Notebook 2 von 3** · läuft **lokal** auf deinem Laptop

```
   P1 Daten sammeln  →  ▶ P2 Labeln & prüfen  →  P3 Training in Colab
                         ────────────────────
     Label Studio · Export · Umrechnen · Kontrolle · Datensatz-ZIP
```

Jetzt bekommen deine Bilder die „richtigen Antworten“: **Boxen mit Klassen**. Danach prüfst du die Labels und
baust daraus den fertigen **Datensatz** für das Training in Colab.

| Abschnitt | Was du tust |
|---|---|
| 1 · Start | Steckbrief laden |
| 2 · Arbeit aufteilen | Bilder auf die Gruppe verteilen (optional) |
| 3 · Label Studio | starten und einrichten, **ohne** Bilder hochzuladen |
| 4 · Labeln | Regeln und Tastenkürzel |
| 5 · Exportieren | JSON-Export pro Person |
| 6 · Zusammenführen | alle Exporte einlesen, Probleme finden |
| 7 · Umrechnen | Label-Studio-Boxen → YOLO-Format (das programmierst du!) |
| 8 · Kontrolle | Statistik, Bilder mit Boxen, Ausschnitte pro Klasse |
| 9 · Datensatz bauen | train/val/test, `data.yaml`, ZIP |
| 10 · Probelauf & Upload | kurzer Test, dann ab in Google Drive |

> 📖 Theorie: [P2 auf der Lernseite](https://rdf-sul.github.io/YOLO_Training/projekt/p2-labeln/) · [Daten labeln](https://rdf-sul.github.io/YOLO_Training/daten/labeln/) ·
> [data.yaml](https://rdf-sul.github.io/YOLO_Training/daten/data-yaml/)

---
## 1 · Start

In [ ]:
%matplotlib inline
import werkzeuge as wz
import pandas as pd

PROJEKT = wz.projekt_laden()
KLASSEN = PROJEKT["klassen"]
print(f"📁 Projekt: {PROJEKT['name']}")
for i, k in enumerate(KLASSEN):
    print(f"   {i} = {k:<12} → {PROJEKT.get('definitionen', {}).get(k, '')}")
print(f"🖼️  Bilder in 2_bilder/: {len(wz.bilder_in(wz.BILDER))}")

---
## 2 · Arbeit in der Gruppe aufteilen *(optional)*

Labeln dauert, pro Bild etwa 20–60 Sekunden. In der Gruppe geht es schneller:

1. **Eine Person** (Projektleitung) führt P1 aus und verteilt hier die Bilder auf Pakete, ein Unterordner pro Person.
2. Der **ganze Projektordner** wird an alle kopiert (USB-Stick, OneDrive, Teams …).
3. Jede Person richtet Label Studio auf **ihrem** Laptop mit **ihrem** Paket ein (Abschnitt 3) und labelt.
4. Jede Person exportiert als JSON (Abschnitt 5) und gibt die Datei der Projektleitung.
5. Die Projektleitung legt alle Exporte in `3_labelstudio_export/` und macht ab Abschnitt 6 weiter.

💡 Das klappt, weil P1 jedem Bild einen **eindeutigen Namen** gegeben hat. P2 ordnet die Labels nur über den Namen zu.
Wo der Ordner auf den einzelnen Laptops liegt, spielt keine Rolle.

Arbeitest du allein, **überspringe** diesen Abschnitt.

🔁 **Neue Bilder beim Nachtraining:** Nach dem Import (P1) liegen neue Bilder lose in `2_bilder/`. Führe diese Zelle
**noch einmal** aus: Sie verteilt nur die neuen Bilder auf die vorhandenen Pakete. Gib die Paketordner wieder an die
Personen. Jede Person klickt in Label Studio bei ihrer Storage auf **Sync Storage**.

In [ ]:
GRUPPE = PROJEKT.get("gruppe", [])     # z. B. ["Anna", "Ben"] – leere Liste [] = keine Aufteilung

if len(GRUPPE) > 1:
    wz.pakete_aufteilen(GRUPPE)
else:
    print("Keine Aufteilung (allein oder keine Gruppe im Steckbrief).")

---
## 3 · Label Studio starten und einrichten

**Label Studio** ist ein kostenloses Labeling-Werkzeug, das **lokal** auf deinem Laptop läuft.
Deine Bilder verlassen den Rechner nicht.

Wir **laden die Bilder nicht hoch**. Label Studio liest sie direkt aus deinem Ordner (*Local Files*).
Das spart Platz, und die Dateinamen bleiben erhalten.

### 3.1 Starten

Die nächste Zelle startet Label Studio im Hintergrund und öffnet es im Browser.
Beim **allerersten** Mal wird es heruntergeladen. Das dauert 1–3 Minuten.

Beim ersten Öffnen legst du ein **lokales Konto** an (*Sign up*). E-Mail und Passwort bleiben auf deinem Laptop.
Merke sie dir.

In [ ]:
wz.labelstudio_starten()

### 3.2 Projekt anlegen und Label-Konfiguration einfügen

Die nächste Zelle zeigt dir zwei Dinge zum **Kopieren**: die **Label-Konfiguration** und den **Ordnerpfad**.

In [ ]:
print("① LABEL-KONFIGURATION  (alles markieren und kopieren):\n")
print(wz.labelstudio_konfiguration(KLASSEN))
print("\n② ORDNERPFAD(E) für „Absolute local path“  (jede Person nur ihr eigenes Paket):\n")
for pfad in wz.ordner_fuer_labelstudio():
    print("   ", pfad)

**In Label Studio** (Version 1.23):

1. **Create Project** (später heißt der Knopf nur **Create**) → Reiter *Project Name*: z. B. `Bauteile – Anna`.
2. Reiter **Labeling Setup** → links ganz unten **Custom template** → es öffnet sich die Ansicht **Code**.
   Den vorhandenen Text `<View></View>` löschen und die **Label-Konfiguration ①** einfügen.
   Rechts in der *Preview* erscheinen deine Klassen mit den Tasten 1, 2, 3 …
3. **Save**. Den Reiter *Data Import* **nicht** benutzen, denn wir laden nichts hoch.

### 3.3 Bilder-Ordner verbinden (*Local Files*)

Im leeren Projekt auf **Connect Cloud Storage** klicken. (Alternativ: **Settings → Cloud Storage → Add Source Storage**.)
Ein Assistent mit vier Schritten öffnet sich:

| Schritt | Eintragen / Klicken |
|---|---|
| ① *Select Provider* | **Local Files** → **Next** |
| ② *Configure Connection* | *Storage Title*: z. B. `Bilder Anna` · *Absolute local path*: **Ordnerpfad ②** einfügen → **Test Connection** → **Next** |
| ③ *Import Settings & Preview* | *Import Method*: **Files – Automatically creates a task for each storage object** (wichtig!) · *File Name Filter*: `.*\.jpg` → **Load Preview** zeigt deine Bilder → **Next** |
| ④ *Review & Confirm* | Anzahl der Dateien prüfen → **Save & Sync** |

Zurück im Projekt (oben auf den Projektnamen klicken) erscheinen jetzt alle Bilder als Aufgaben (*Tasks*).

💡 **Fehlermeldung „… is not presented in LOCAL_FILES_DOCUMENT_ROOT“?** Label Studio wurde nicht über die Zelle oben
gestartet, oder der Pfad liegt außerhalb des Projektordners. Label Studio beenden (Abschnitt 5) und neu starten.

---
## 4 · Labeln

**Label All Tasks** öffnet das erste Bild.

| Aktion | So geht's |
|---|---|
| Klasse wählen | Taste <kbd>1</kbd>, <kbd>2</kbd>, <kbd>3</kbd> … (oder Klick auf den Namen unter dem Bild) |
| Box zeichnen | mit der Maus aufziehen |
| Box korrigieren | anklicken → an den Ecken ziehen · <kbd>Entf</kbd> löscht |
| Zoomen | Mausrad / <kbd>Strg</kbd> + Mausrad |
| **fertig → nächstes Bild** | **Submit** (<kbd>Strg</kbd>+<kbd>Enter</kbd>) |
| Bild ist unbrauchbar | **Skip**: Es wird beim Training **nicht** verwendet |

### Die Label-Regeln (für die ganze Gruppe gleich!)

1. **Jedes** Objekt jeder Klasse labeln, auch angeschnittene und teilweise verdeckte.
   Ein vergessenes Objekt bringt dem Modell bei: „Das ist Hintergrund.“
2. **Eng** umrahmen: kein Rand mit viel Hintergrund, aber nichts abschneiden.
3. Halte dich an die **Definitionen aus dem Steckbrief** (Abschnitt 1), z. B. ob Anschlussdrähte zur Box gehören.
4. Bild **ohne** Objekte (Hintergrundbild)? → einfach **Submit** ohne Box. Das ist ausdrücklich erlaubt.
5. Unsicher, was es ist (zu klein, zu unscharf)? → **Skip** statt raten.

⚠️ **Submit ohne Box** bedeutet: „Auf diesem Bild ist nichts.“ Abschnitt 8 zeigt dir alle solchen Bilder.
Prüfe dort, ob du nicht aus Versehen etwas vergessen hast.

---
## 5 · Exportieren

1. In der Aufgabenliste des Projekts oben rechts **Export**.
2. Format **JSON** (ist vorausgewählt, **nicht** JSON-MIN, **nicht** YOLO) → unten **Export**.
3. Die heruntergeladene Datei (`project-1-at-2026-…json`) umbenennen in **`<dein_name>.json`**, z. B. `anna.json`.
4. Die Datei nach **`3_labelstudio_export/`** legen (bzw. der Projektleitung geben). Eine ältere Datei mit gleichem Namen ersetzen.

**Warum JSON und nicht YOLO?** Der YOLO-Export von Label Studio enthält bei *Local Files* **keine Bilder**, und je nach
Version passen die Dateinamen nicht zu den Bildern. Der JSON-Export enthält dagegen zu jeder Box den **Originalpfad
des Bildes**. Daraus machen wir in Abschnitt 7 selbst sauberes YOLO-Format.

🔁 **Später korrigieren:** In Label Studio ändern → neu exportieren → **alte Datei überschreiben** → ab Abschnitt 6 neu ausführen.

Label Studio beenden (wenn du fertig bist):

In [ ]:
wz.labelstudio_stoppen()

---
## 6 · Exporte einlesen und zusammenführen

Die nächste Zelle liest **alle** `.json`-Dateien aus `3_labelstudio_export/` und führt sie zusammen. Sie prüft dabei:

| Prüfung | Was bedeutet es? | Was tun? |
|---|---|---|
| **doppelt gelabelt / KONFLIKT** | Ein Bild steht in zwei Exporten (z. B. zwei Personen) | Die **neueste** Version wird genommen. Bei KONFLIKT beide ansehen! |
| **Bild fehlt** | Label existiert, Bild nicht (mehr) in `2_bilder/` | Bild aussortiert? Dann ok. Sonst wiederherstellen |
| **Bildgröße passt nicht** | Bild wurde **nach** dem Labeln verändert (z. B. gedreht) | Bild in Label Studio neu labeln |
| **Unbekannte Klasse** | Klassenname in Label Studio ≠ Steckbrief | Label-Konfiguration prüfen (Abschnitt 3.2) |
| **Falsches Format** | Export war JSON-MIN oder YOLO | Neu exportieren als **JSON** |

In [ ]:
annotationen, protokoll = wz.exporte_einlesen(KLASSEN)
if len(protokoll):
    display(protokoll)

---
## 7 · Umrechnen ins YOLO-Format ✏️

Label Studio und YOLO beschreiben eine Box **unterschiedlich**:

| | Label Studio | YOLO |
|---|---|---|
| Bezugspunkt | **linke obere Ecke** `x`, `y` | **Mittelpunkt** `x_mitte`, `y_mitte` |
| Größe | `width`, `height` | `breite`, `hoehe` |
| Einheit | **Prozent** (0 … 100) der Bildbreite bzw. -höhe | **Anteil** (0 … 1) der Bildbreite bzw. -höhe |

```
 Label Studio:  (x, y) ●──────── width ────────┐        YOLO:  ┌───────────────────────┐
                       │                       │               │                       │
                     height                    │               │      ● (x_mitte, y_mitte)
                       │                       │               │                       │
                       └───────────────────────┘               └───────────────────────┘
```

**Deine Aufgabe:** Ergänze in der nächsten Zelle die beiden Zeilen mit `...`. Die Testzeile darunter prüft dein Ergebnis.
Solange die Funktion nicht stimmt, meldet sie einen Fehler. Alle weiteren Zellen brauchen diese Funktion.

1. Rechne zuerst **von Hand**: Eine Box hat in Label Studio `x=40, y=10, width=20, height=30`. Wie lautet die YOLO-Zeile?
2. Programmiere dann die beiden fehlenden Zeilen. Tipp: Mittelpunkt = linke Kante + **halbe** Breite.
3. **Experiment:** Vertausche danach absichtlich `breite` und `hoehe` in der `return`-Zeile. Was meldet der Test?
   Mach es wieder rückgängig. Solche **Tests** schützen in echter Software vor Fehlern.
4. *Für Schnelle:* Rechne die YOLO-Box `0.5 0.25 0.2 0.3` für ein Bild mit 1280 × 720 Pixeln zurück in **Pixel**
   (linke obere Ecke, Breite, Höhe).

Nicht weitergekommen? Die Lösung steht auf der Lernseite unter **P2 · 7 · Umrechnen ins YOLO-Format**.

In [ ]:
def umrechnen(box):
    """Label-Studio-Box (Prozent, linke obere Ecke) → YOLO-Box (0…1, Mittelpunkt)."""
    x      = box["x"] / 100          # linke Kante als Anteil der Bildbreite
    y      = box["y"] / 100          # obere Kante als Anteil der Bildhöhe
    breite = box["width"] / 100
    hoehe  = box["height"] / 100
    x_mitte = ...                    # ✏️ Mittelpunkt in x-Richtung
    y_mitte = ...                    # ✏️ Mittelpunkt in y-Richtung
    return x_mitte, y_mitte, breite, hoehe


# ✔️ Test: Box von 40 % bis 60 % in der Breite und von 10 % bis 40 % in der Höhe
ergebnis = umrechnen({"x": 40, "y": 10, "width": 20, "height": 30})
erwartet = (0.5, 0.25, 0.2, 0.3)
assert all(isinstance(a, float) and abs(a - b) < 1e-9 for a, b in zip(ergebnis, erwartet)), \
    f"Noch nicht richtig: {ergebnis} – erwartet {erwartet}"
print("✅ Umrechnung stimmt:", ergebnis)

Jetzt wird für **jedes gelabelte Bild** eine YOLO-Textdatei geschrieben, eine Zeile pro Box:
`klasse x_mitte y_mitte breite hoehe`. Die Klasse wird dabei vom Namen zur **Nummer** (Reihenfolge im Steckbrief).

In [ ]:
wz.labels_schreiben(annotationen, KLASSEN, umrechnen)
LABELS = wz.labels_lesen()

beispiel = next((n for n, b in LABELS.items() if b), next(iter(LABELS)))   # erstes Bild mit Boxen
print(f"\nBeispiel 4_datensatz/labels/{beispiel[:-4]}.txt:")
print((wz.DATENSATZ / "labels" / (beispiel[:-4] + ".txt")).read_text())

---
## 8 · Kontrolle: Sind die Labels gut?

> **„Garbage in, garbage out“:** Fehlerhafte Labels schaden mehr als fehlende Bilder.
> Ein Modell kann nie besser werden als seine Labels.

### 8.1 Statistik: Reicht die Menge?

In [ ]:
tabelle = wz.statistik(LABELS, KLASSEN, PROJEKT.get("ziel_objekte_pro_klasse"))
display(tabelle)

**So liest du die Diagramme:**

- **Objekte pro Klasse:** Erreicht jede Klasse den Zielwert (gestrichelte Linie)? Sind die Klassen ähnlich häufig?
- **Objekte pro Bild:** Eine Säule bei 0 zeigt die Hintergrundbilder (ideal: 5–10 % aller Bilder).
- **Objektgröße:** Liegen viele Objekte im grauen Bereich (< 5 % der Bildseite), werden sie beim Training mit 640 px
  sehr klein. Dann lieber näher fotografieren oder später in P3 mit `BILDGROESSE = 1280` trainieren.

✏️ **Vermutung notieren:** Welche Klasse wird euer Modell vermutlich **am schlechtesten** erkennen? Begründet es mit der
Statistik (Anzahl, Größe) oder später mit den Ausschnitten. In P3 prüft ihr, ob die Vermutung stimmt.

### 8.2 Bilder mit Boxen durchsehen

Wähle oben einen **Filter**, z. B. eine Klasse oder „nur Hintergrundbilder“. Achte auf:
vergessene Objekte · falsche Klasse · zu große/kleine Boxen · doppelte Boxen.

In [ ]:
wz.labelbrowser(LABELS, KLASSEN)

### 8.3 Ausschnitte pro Klasse

Hier siehst du **alle Objekte einer Klasse nebeneinander**. Ein falsch gelabeltes Objekt fällt zwischen lauter
richtigen sofort auf, ebenso **uneinheitliche** Boxen (mal mit, mal ohne Anschlussdrähte).
Den Bildnamen zeigt die Maus beim Darüberfahren.

In [ ]:
for k in KLASSEN:
    wz.ausschnitte(LABELS, KLASSEN, k, max_anzahl=40)

🔁 **Fehler gefunden?** Bildname notieren → in Label Studio das Bild suchen (Suchfeld/Filter) → korrigieren → **Submit** →
neu exportieren (Abschnitt 5) → ab Abschnitt 6 neu ausführen.

---
## 9 · Datensatz bauen

Für das Training werden die Bilder in drei **Teilmengen** aufgeteilt:

| Teilmenge | Anteil | Wozu? | Vergleich |
|---|---|---|---|
| **train** | 70 % | daraus lernt das Modell | Übungsaufgaben |
| **val** | 20 % | Kontrolle **während** des Trainings, Auswahl von `best.pt` | Probeklausur |
| **test** | 10 % | **einmalige** Abschlussbewertung, nie zum Entscheiden benutzt | echte Prüfung |

Zwei Besonderheiten machen die Aufteilung **fair** und **nachtrainings-fest**:

1. **Einmal zugeteilt, immer zugeteilt.** Die Zuteilung wird in `4_datensatz/aufteilung.csv` gespeichert.
   Kommen später neue Bilder dazu (Version `v2`), bleiben alle alten Bilder in ihrer Teilmenge.
   Nur so lassen sich Version 1 und Version 2 ehrlich vergleichen.
2. **Serienfotos bleiben zusammen.** Fast gleiche Bilder kommen in **dieselbe** Teilmenge. Läge ein Foto in *train* und
   sein Zwilling in *val*, würde das Modell in der Probeklausur Aufgaben sehen, die es schon kennt. Die Bewertung wäre zu gut.

Neue Bilder werden so verteilt, dass jede Teilmenge ihren Anteil bekommt und **jede Klasse möglichst auch in val
und test vorkommt**. Klappt das bei sehr seltenen Klassen nicht, erscheint eine Warnung.

✏️ **Version:** `v1` für den ersten Datensatz. Bei jeder späteren Erweiterung hochzählen: `v2`, `v3` …

In [ ]:
VERSION = "v1"                  # ← bei jeder Erweiterung hochzählen: v2, v3 …
ANTEILE = (0.7, 0.2, 0.1)       # train, val, test
NOTIZ   = "Erster Datensatz"    # ← Was ist neu in dieser Version? (z. B. "+80 Bilder Taster bei Tageslicht")

zuteilung = wz.aufteilen(LABELS, ANTEILE)
DATENSATZ_ORDNER = wz.datensatz_bauen(LABELS, KLASSEN, zuteilung, VERSION, NOTIZ)

Die **`data.yaml`** sagt Ultralytics, wo die Bilder liegen und wie die Klassen heißen.
Weil kein `path:` darin steht, gelten die Pfade **relativ zum Ordner der data.yaml**.
Deshalb funktioniert derselbe Datensatz auf deinem Laptop und in Colab.

In [ ]:
print((DATENSATZ_ORDNER / "data.yaml").read_text(encoding="utf-8"))

---
## 10 · Probelauf und Upload

### 10.1 Probelauf *(1–2 Minuten)*

Bevor du in Colab eine halbe Stunde trainierst, prüfe kurz hier: **Kann Ultralytics den Datensatz lesen?**
Die Zelle trainiert nur **eine** Epoche mit kleinen Bildern. Das Ergebnis ist unwichtig, es zählt nur, dass **kein Fehler** kommt.

In [ ]:
from ultralytics import YOLO

PROBELAUF = True   # auf False setzen, um ihn zu überspringen

if PROBELAUF:
    modell = YOLO("yolo26n.pt")
    modell.train(data=str(DATENSATZ_ORDNER / "data.yaml"), epochs=1, imgsz=320, batch=8, workers=0,
                 project=str(wz.DATENSATZ.resolve() / "probelauf"), name=VERSION, exist_ok=True,
                 plots=False, verbose=False)
    print("\n✅ Probelauf erfolgreich – der Datensatz ist bereit für Colab.")

### 10.2 Hochladen in Google Drive

1. [drive.google.com](https://drive.google.com) öffnen → **Meine Ablage**.
2. Ordner **`YOLO_Projekt`** anlegen und darin den Ordner **`datensaetze`**.
3. Die ZIP-Datei aus `4_datensatz/` (z. B. `datensatz_v1.zip`) in `YOLO_Projekt/datensaetze/` hochladen.

```
Meine Ablage/
└── YOLO_Projekt/
    ├── datensaetze/
    │   └── datensatz_v1.zip      ← hochladen
    ├── laeufe/                   ← legt P3 an (Trainingsergebnisse, auch nach Abbruch)
    └── modelle/                  ← legt P3 an (deine fertigen Modelle)
```

### ✅ Checkliste vor P3

- [ ] Alle Bilder gelabelt oder bewusst übersprungen, Protokoll ohne offene Probleme
- [ ] Jede Klasse hat möglichst den Zielwert erreicht, Klassen ungefähr ausgewogen
- [ ] Labels durchgesehen: Ausschnitte pro Klasse einheitlich, Hintergrundbilder wirklich leer
- [ ] Datensatz gebaut, Probelauf ohne Fehler
- [ ] ZIP liegt in Google Drive unter `YOLO_Projekt/datensaetze/`

➡️ **Weiter mit `P3_training_colab.ipynb`** – in Google Colab öffnen: [P3 auf der Lernseite](https://rdf-sul.github.io/YOLO_Training/projekt/p3-training/)